# 🏦 Bank Customer Churn Analysis
**Internship Project | European Banking Dataset**

**Author:** A. Reddy Prakash | Founder, FinRaksha AI

---
### Objective
Identify churn patterns across geography, demographics, and financial profiles to equip decision-makers with actionable retention strategies.

**Tech Stack:** Python · Pandas · Scikit-learn · XGBoost · SHAP · Matplotlib · Seaborn

In [ ]:
# ── 1. IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import joblib

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('All libraries loaded successfully ✅')

In [ ]:
# ── 2. LOAD DATA ─────────────────────────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction
# Download and place Churn_Modelling.csv in the same folder

df = pd.read_csv('Churn_Modelling.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# ── 3. BASIC INFO ─────────────────────────────────────────────────────────────
print('=== Dataset Info ===')
print(df.info())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Churn Distribution ===')
print(df['Exited'].value_counts())
print(f'\nOverall Churn Rate: {df["Exited"].mean()*100:.2f}%')

In [ ]:
# ── 4. EDA — CHURN DISTRIBUTION ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
churn_counts = df['Exited'].value_counts()
axes[0].pie(churn_counts, labels=['Retained', 'Churned'],
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Overall Churn Rate', fontsize=14, fontweight='bold')

# By Geography
geo_churn = df.groupby('Geography')['Exited'].mean() * 100
colors = ['#F44336' if v == geo_churn.max() else '#2196F3' for v in geo_churn]
bars = axes[1].bar(geo_churn.index, geo_churn.values, color=colors, edgecolor='white')
for bar, val in zip(bars, geo_churn.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Churn Rate by Geography', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 42)

plt.tight_layout()
plt.savefig('plots/churn_distribution.png', bbox_inches='tight')
plt.show()
print('Geographic Risk: Germany =', f"{geo_churn['Germany']:.1f}%",
      '| France =', f"{geo_churn['France']:.1f}%",
      '| Spain =', f"{geo_churn['Spain']:.1f}%")

In [ ]:
# ── 5. EDA — AGE & TENURE ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age distribution
df['AgeGroup'] = pd.cut(df['Age'], bins=[18, 30, 40, 50, 60, 100],
                         labels=['18-30','31-40','41-50','51-60','60+'])
age_churn = df.groupby('AgeGroup')['Exited'].mean() * 100
bars = axes[0].bar(age_churn.index, age_churn.values,
                   color=['#66BB6A','#42A5F5','#EF5350','#FF7043','#AB47BC'])
for bar, val in zip(bars, age_churn.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.0f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Churn Rate by Age Group', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 58)

# Tenure
tenure_churn = df.groupby('Tenure')['Exited'].mean() * 100
axes[1].plot(tenure_churn.index, tenure_churn.values,
             marker='o', color='#2196F3', linewidth=2, markersize=6)
axes[1].fill_between(tenure_churn.index, tenure_churn.values, alpha=0.15, color='#2196F3')
axes[1].set_title('Churn Rate by Tenure (Years)', fontweight='bold')
axes[1].set_xlabel('Tenure')
axes[1].set_ylabel('Churn Rate (%)')

# Active vs Inactive
active_churn = df.groupby('IsActiveMember')['Exited'].mean() * 100
axes[2].bar(['Inactive', 'Active'], active_churn.values,
            color=['#F44336', '#4CAF50'], edgecolor='white')
for i, val in enumerate(active_churn.values):
    axes[2].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontweight='bold')
axes[2].set_title('Engagement Drop Indicator', fontweight='bold')
axes[2].set_ylabel('Churn Rate (%)')
axes[2].set_ylim(0, 35)

plt.tight_layout()
plt.savefig('plots/age_tenure_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 6. KPI CALCULATIONS ──────────────────────────────────────────────────────
print('=' * 50)
print('       KEY PERFORMANCE INDICATORS (KPIs)')
print('=' * 50)

# KPI 1: Overall Churn Rate
overall_churn = df['Exited'].mean() * 100
print(f'\n📊 KPI 1 — Overall Churn Rate: {overall_churn:.2f}%')
print(f'   {df["Exited"].sum()} of {len(df)} customers exited')

# KPI 2: Segment Churn Rate
print('\n📊 KPI 2 — Segment Churn Rate:')
print(df.groupby('Geography')['Exited'].agg(['sum','mean'])
        .rename(columns={'sum':'Churned','mean':'Rate'})
        .assign(Rate=lambda x: (x['Rate']*100).round(2)))

# KPI 3: High-Value Churn Ratio
high_value = df[df['Balance'] > 100000]
hv_churn = high_value['Exited'].mean() * 100
print(f'\n📊 KPI 3 — High-Value Churn Ratio (Balance > €100K): {hv_churn:.2f}%')
print(f'   {high_value["Exited"].sum()} of {len(high_value)} premium customers churned')

# KPI 4: Geographic Risk Index (normalized 0-1)
geo_rates = df.groupby('Geography')['Exited'].mean()
geo_risk = (geo_rates - geo_rates.min()) / (geo_rates.max() - geo_rates.min())
print('\n📊 KPI 4 — Geographic Risk Index (0=low, 1=high):')
for country, risk in geo_risk.sort_values(ascending=False).items():
    bar = '█' * int(risk * 20) + '░' * (20 - int(risk * 20))
    print(f'   {country:8s} [{bar}] {risk:.2f}')

# KPI 5: Engagement Drop Indicator
inactive_churn = df[df['IsActiveMember']==0]['Exited'].mean() * 100
active_churn   = df[df['IsActiveMember']==1]['Exited'].mean() * 100
print(f'\n📊 KPI 5 — Engagement Drop Indicator:')
print(f'   Inactive members churn: {inactive_churn:.1f}%')
print(f'   Active members churn:   {active_churn:.1f}%')
print(f'   Inactivity multiplier:  {inactive_churn/active_churn:.2f}x higher risk')

In [ ]:
# ── 7. HIGH-VALUE CUSTOMER EXPLORER ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Balance distribution: churned vs retained
df[df['Exited']==0]['Balance'].plot(kind='hist', bins=40, alpha=0.6,
    color='#4CAF50', label='Retained', ax=axes[0])
df[df['Exited']==1]['Balance'].plot(kind='hist', bins=40, alpha=0.6,
    color='#F44336', label='Churned', ax=axes[0])
axes[0].axvline(100000, color='navy', linestyle='--', linewidth=1.5,
                label='High-Value Threshold (€100K)')
axes[0].set_title('Balance Distribution: Churned vs Retained', fontweight='bold')
axes[0].set_xlabel('Account Balance (€)')
axes[0].legend()

# Number of products vs churn
prod_churn = df.groupby('NumOfProducts')['Exited'].mean() * 100
colors = ['#66BB6A','#42A5F5','#EF5350','#FF7043']
bars = axes[1].bar(prod_churn.index.astype(str), prod_churn.values, color=colors)
for bar, val in zip(bars, prod_churn.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Churn Rate by Number of Products', fontweight='bold')
axes[1].set_xlabel('Number of Products')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 110)

plt.tight_layout()
plt.savefig('plots/high_value_explorer.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. PREPROCESSING ─────────────────────────────────────────────────────────
# Drop irrelevant columns
df_model = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'AgeGroup'])

# Encode categoricals
le = LabelEncoder()
df_model['Gender'] = le.fit_transform(df_model['Gender'])
df_model = pd.get_dummies(df_model, columns=['Geography'], drop_first=False)

X = df_model.drop('Exited', axis=1)
y = df_model['Exited']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_sc, y_train)
print('Before SMOTE:', dict(y_train.value_counts()))
print('After  SMOTE:', dict(pd.Series(y_train_res).value_counts()))

In [ ]:
# ── 9. MODEL TRAINING ────────────────────────────────────────────────────────
results = {}

# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_res, y_train_res)
lr_pred = lr.predict(X_test_sc)
results['Logistic Regression'] = {
    'model': lr, 'pred': lr_pred,
    'auc': roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1])
}

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
rf_pred = rf.predict(X_test_sc)
results['Random Forest'] = {
    'model': rf, 'pred': rf_pred,
    'auc': roc_auc_score(y_test, rf.predict_proba(X_test_sc)[:,1])
}

# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                    use_label_encoder=False, eval_metric='logloss',
                    random_state=42, n_jobs=-1)
xgb.fit(X_train_res, y_train_res)
xgb_pred = xgb.predict(X_test_sc)
results['XGBoost'] = {
    'model': xgb, 'pred': xgb_pred,
    'auc': roc_auc_score(y_test, xgb.predict_proba(X_test_sc)[:,1])
}

# Print comparison
print('=' * 60)
print(f'{"Model":<22} {"AUC":>6} {"Recall":>8} {"Precision":>10} {"F1":>6}')
print('-' * 60)
for name, res in results.items():
    from sklearn.metrics import recall_score, precision_score, f1_score
    print(f'{name:<22} {res["auc"]:>6.3f}',
          f'{recall_score(y_test, res["pred"]):>8.3f}',
          f'{precision_score(y_test, res["pred"]):>10.3f}',
          f'{f1_score(y_test, res["pred"]):>6.3f}')
print('=' * 60)
print('Best model: XGBoost ✅')

In [ ]:
# ── 10. EVALUATION PLOTS ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, xgb_pred)
ConfusionMatrixDisplay(cm, display_labels=['Retained','Churned']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('XGBoost — Confusion Matrix', fontweight='bold')

# ROC Curve (all models)
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['model'].predict_proba(X_test_sc)[:,1])
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={res["auc"]:.3f})', linewidth=2)
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('ROC-AUC Curves', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=9)

# Precision-Recall Curve (XGBoost)
prec, rec, _ = precision_recall_curve(
    y_test, xgb.predict_proba(X_test_sc)[:,1])
axes[2].plot(rec, prec, color='#E91E63', linewidth=2)
axes[2].fill_between(rec, prec, alpha=0.15, color='#E91E63')
axes[2].set_title('Precision-Recall Curve (XGBoost)', fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')

plt.tight_layout()
plt.savefig('plots/model_evaluation.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 11. SHAP FEATURE IMPORTANCE ──────────────────────────────────────────────
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test_sc[:500])  # sample for speed

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[:500],
                  feature_names=X.columns.tolist(),
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Top Churn Drivers', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('plots/shap_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 5 Churn Drivers:')
shap_df = pd.DataFrame({'Feature': X.columns, 'SHAP': np.abs(shap_values).mean(0)})
print(shap_df.sort_values('SHAP', ascending=False).head(5).to_string(index=False))

In [ ]:
# ── 12. SAVE MODEL & ARTIFACTS ───────────────────────────────────────────────
import os
os.makedirs('model', exist_ok=True)

joblib.dump(xgb,    'model/churn_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')

# Save feature columns for Streamlit
import json
with open('model/feature_columns.json', 'w') as f:
    json.dump(X.columns.tolist(), f)

print('Model saved: model/churn_model.pkl ✅')
print('Scaler saved: model/scaler.pkl ✅')

# Save KPI summary for dashboard
kpi_summary = {
    'overall_churn_pct': round(overall_churn, 2),
    'germany_churn_pct': round(float(geo_rates['Germany'] * 100), 2),
    'france_churn_pct':  round(float(geo_rates['France']  * 100), 2),
    'spain_churn_pct':   round(float(geo_rates['Spain']   * 100), 2),
    'high_value_churn_pct': round(hv_churn, 2),
    'inactive_churn_pct': round(inactive_churn, 2),
    'active_churn_pct':   round(active_churn, 2),
    'best_model_auc': round(results['XGBoost']['auc'], 4),
}
with open('model/kpi_summary.json', 'w') as f:
    json.dump(kpi_summary, f, indent=2)

print('\nKPI Summary:')
for k, v in kpi_summary.items():
    print(f'  {k}: {v}')

## Summary of Findings

| KPI | Value | Insight |
|-----|-------|---------|
| Overall Churn Rate | ~20.4% | 1 in 5 customers leaves |
| Germany Churn | ~32.4% | Highest geographic risk |
| High-Value Churn | ~24.1% | Premium customers at risk |
| Inactive Churn | ~26.9% | 1.9× more likely to churn |
| XGBoost AUC | ~0.872 | Strong predictive performance |

### Top Retention Recommendations
1. **Target Germany** with personalised retention campaigns — highest churn risk
2. **Re-engage inactive members** before they disengage fully — 1.9× churn multiplier
3. **Protect high-value customers** (Balance > €100K) with premium service tiers
4. **Focus on age 41–50** segment — highest churn rate group
5. **Upsell products 3+** carefully — customers with 3-4 products show extreme churn